In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("workspace.bronze.products")

df = (
    df
    .withColumn("price", F.col("price").cast("decimal(12,2)"))
    .withColumn("cost", F.col("cost").cast("decimal(12,2)"))
    .withColumn("updated_at",F.to_timestamp("updated_at", "dd-MM-yyyy HH:mm"))
    .withColumn("product_name", F.trim("product_name"))
)
df.show()

In [0]:
window = (
    Window
    .partitionBy("product_id")
    .orderBy(F.col("updated_at").desc())
)

silver_products = (
    df
    .withColumn("rn", F.row_number().over(window))
    .filter("rn = 1")
    .drop("rn")
    .filter(F.col("price") >= 0)
)

(
    silver_products.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.products")
)



In [0]:
display(silver_products)